In [10]:
import requests
import pandas as pd
import urllib3
import os

# Ignorar alertas SSL de servidores governamentais
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
base_url = "https://scpi.assis.sp.gov.br:8079/transparencia"

print("Iniciando auditoria de dados da Prefeitura de Assis-SP...\n")

# 1. EXTRAÇÃO DE DADOS (EXTRACT)
print("Coletando despesas gerais do Exercício 2026...")
url_despesas = f"{base_url}/VersaoJson/Despesas/?ConectarExercicio=2026&Listagem=DespesasPorFornecedor&DiaInicioPeriodo=01&MesInicialPeriodo=01&DiaFinalPeriodo=31&MesFinalPeriodo=12&Ano=2026&Empresa=2&MostrarFornecedor=True&MostraDadosConsolidado=False"

try:
    res_despesas = requests.get(url_despesas, verify=False, timeout=20)
    df_despesas = pd.DataFrame(res_despesas.json())
    print(f"Sucesso: {len(df_despesas)} fornecedores totais extraídos da API.\n")

    # Salvando RAW
    os.makedirs('../data/raw', exist_ok=True)
    df_despesas.to_csv('../data/raw/despesas_ti_assis_2026_bruto.csv', index=False, encoding='utf-8')
    print("RAW salvo em: '../data/raw/despesas_ti_assis_2026_bruto.csv'\n")

    # 2. PROCESSAMENTO E MINERAÇÃO (TRANSFORM)
    print("Minerando contratos de Tecnologia (Filtro Regex)...")
    palavras_chave = 'SOFTWARE|INFORMATICA|LICENCA|MICROSOFT|SISTEMA|TECNOLOGIA'
    df_ti = df_despesas[df_despesas['DESCRICAO'].str.contains(palavras_chave, case=False, na=False)].copy()

    # Limpeza e conversão de valores financeiros
    df_ti['EMPENHADO'] = pd.to_numeric(df_ti['EMPENHADO'].astype(str).str.replace(',', '.'), errors='coerce')
    df_ti['PAGO'] = pd.to_numeric(df_ti['PAGO'].astype(str).str.replace(',', '.'), errors='coerce')

    # Ordenando pelos maiores contratos
    df_ti = df_ti.sort_values(by='EMPENHADO', ascending=False)
    print(f"Sucesso: {len(df_ti)} fornecedores de TI/Sistemas identificados.\n")

    # 3. ANÁLISE DE RESULTADOS E ANOMALIAS
    print("FORNECEDORES ENCONTRADOS (ORDEM DE EMPENHO, max:20):")
    print(df_ti[['DESCRICAO', 'EMPENHADO', 'PAGO']].head(20).to_string(index=False))

    print("\nFALSOS POSITIVOS:")
    df_anomalias = df_ti[df_ti['DESCRICAO'].str.contains('ILUMINAÇÃO|ENERGIA|STYLUX', case=False, na=False)]
    if not df_anomalias.empty:
        print(df_anomalias[['DESCRICAO', 'EMPENHADO']].to_string(index=False))
        
        # Calculando o impacto da anomalia
        impacto = df_anomalias['EMPENHADO'].sum()
        total_bruto = df_ti['EMPENHADO'].sum()
        print(f"\nImpacto financeiro do falso positivo: R$ {impacto:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
        print(f"Total bruto atual (com erro): R$ {total_bruto:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
    else:
        print("Nenhum falso positivo.")

except Exception as e:
    print(f"Erro crítico na execução: {e}")


Iniciando auditoria de dados da Prefeitura de Assis-SP...

Coletando despesas gerais do Exercício 2026...
Sucesso: 1979 fornecedores totais extraídos da API.

RAW salvo em: '../data/raw/despesas_ti_assis_2026_bruto.csv'

Minerando contratos de Tecnologia (Filtro Regex)...
Sucesso: 37 fornecedores de TI/Sistemas identificados.

FORNECEDORES ENCONTRADOS (ORDEM DE EMPENHO, max:20):
                                         DESCRICAO  EMPENHADO       PAGO
 STYLUX GREENTECH SISTEMAS DE ILUMINAÇÃO E ENERGIA 3372582.80 2081329.44
                 AMENDOLA & AMENDOLA SOFTWARE LTDA  646031.98  173051.64
                    INTERMAPAS GEOTECNOLOGIAS LTDA  517515.46  196996.00
      DEMANDANET DESENVOLVIMENTO DE SOFTWARES LTDA  390000.00  156000.00
                             CLONE TECNOLOGIA LTDA  235289.20   53849.36
                DSIN TECNOLOGIA DA INFORMACAO LTDA   72466.14   36233.07
                       ARLEI ANDREOTTI INFORMATICA   69111.00   42280.00
  WARELINE BRASIL DESENVOLVIMENTO 

In [11]:
import os

# 4. TRATAMENTO DE ANOMALIAS E CÁLCULO DE KPIs
print("\nAPLICANDO TRATAMENTO E ORDENAÇÃO")

# 1. Removendo o falso positivo
df_ti_limpo = df_ti[~df_ti['DESCRICAO'].str.contains('ILUMINAÇÃO|ENERGIA|STYLUX', case=False, na=False)].copy()

# 2. Ordenando do MAIOR para o MENOR valor empenhado
df_ti_limpo = df_ti_limpo.sort_values(by='EMPENHADO', ascending=False)

# Consolidação dos Dados Financeiros Reais
gasto_total_real = df_ti_limpo['EMPENHADO'].sum()

# CAMADA DE APRESENTAÇÃO: VISUALIZAÇÃO NO TERMINAL (HUMAN-READABLE)
print("\nFORNECEDORES REAIS DE TI (ORDENADOS POR EMPENHO, max:20):")

# df_visual apenas para o print, preservando o df_ti_limpo original
df_visual = df_ti_limpo[['DESCRICAO', 'EMPENHADO', 'PAGO']].head(20).copy()
df_visual['EMPENHADO'] = df_visual['EMPENHADO'].apply(lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
df_visual['PAGO'] = df_visual['PAGO'].apply(lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

print(df_visual.to_string(index=False))

# Modelagem Estatística
total_estacoes_estimadas = 1200
taxa_elegibilidade_os = 0.60 
estacoes_elegiveis = int(total_estacoes_estimadas * taxa_elegibilidade_os)

# Cálculo dos KPIs
clu_atual = gasto_total_real / total_estacoes_estimadas
custos_transicao_estimados = 50000.00
economia_projetada = (estacoes_elegiveis * clu_atual) - custos_transicao_estimados

# Formatação pt-BR(R$)
print("\nRESULTADOS FINAIS DOS KPIs:")
print(f"Gasto Real em TI (sem falso positivo): R$ {gasto_total_real:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f"Custo de Licença por Usuário (CLU): R$ {clu_atual:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f"Economia Projetada (1º Ano): R$ {economia_projetada:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

# CAMADA DE DADOS: SALVANDO O CSV (MACHINE-READABLE)
# Colunas financeiras em floats para EXPORT
df_ti_limpo['EMPENHADO'] = df_ti_limpo['EMPENHADO'].astype(float)
df_ti_limpo['PAGO'] = df_ti_limpo['PAGO'].astype(float)
df_ti_limpo['LIQUIDADO'] = pd.to_numeric(df_ti_limpo['LIQUIDADO'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)

# Salvando PROCESSED
os.makedirs('../data/processed', exist_ok=True)
df_ti_limpo.to_csv('../data/processed/despesas_ti_assis_2026_limpo.csv', index=False, encoding='utf-8')
print("\nPROCESSED salvo em: '../data/processed/despesas_ti_assis_2026_limpo.csv'.")


APLICANDO TRATAMENTO E ORDENAÇÃO

FORNECEDORES REAIS DE TI (ORDENADOS POR EMPENHO, max:20):
                                         DESCRICAO     EMPENHADO          PAGO
                 AMENDOLA & AMENDOLA SOFTWARE LTDA R$ 646.031,98 R$ 173.051,64
                    INTERMAPAS GEOTECNOLOGIAS LTDA R$ 517.515,46 R$ 196.996,00
      DEMANDANET DESENVOLVIMENTO DE SOFTWARES LTDA R$ 390.000,00 R$ 156.000,00
                             CLONE TECNOLOGIA LTDA R$ 235.289,20  R$ 53.849,36
                DSIN TECNOLOGIA DA INFORMACAO LTDA  R$ 72.466,14  R$ 36.233,07
                       ARLEI ANDREOTTI INFORMATICA  R$ 69.111,00  R$ 42.280,00
  WARELINE BRASIL DESENVOLVIMENTO DE SOFTWARE LTDA  R$ 65.135,16  R$ 12.255,86
              NOBRITECH TECNOLOGIA E SISTEMAS LTDA  R$ 62.400,00  R$ 31.200,00
ORDEM PÚBLICA, CONSULTORIA, SOFTWARE E TREINAMENTO  R$ 61.980,00  R$ 25.140,00
                          SMARAPD INFORMATICA LTDA  R$ 60.775,00  R$ 55.983,40
                    JVM - PROJETOS E S